# Week 1–2 — Data Exploration

Preview the World Cup 2026 corpus end-to-end:

1. Raw CSVs
2. SQLite load via `csv_loader.load_all_csvs`
3. Bilingual chunks via `text_converter.convert_all`
4. Joined `matches_full` table (schedule × probabilities)
5. Wikipedia corpus (5 articles × EN/AR)
6. Synthetic Q&A samples

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sqlalchemy import create_engine

from src.config import settings
from src.data.csv_loader import (
    HOST_CITIES_CSV,
    PROBABILITIES_CSV,
    SCHEDULE_CSV,
    load_all_csvs,
)
from src.data.match_joiner import build_matches_full
from src.data.text_converter import convert_all

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("CSV dir:    ", settings.paths.csvs)
print("SQLite DB:  ", settings.paths.sqlite_db)
print("Wikipedia:  ", settings.paths.wikipedia)
print("Synthetic:  ", settings.paths.synthetic)

CSV dir:     C:\Users\MDK\worldcup-rag-chatbot\data\raw\csvs
SQLite DB:   C:\Users\MDK\worldcup-rag-chatbot\data\processed\worldcup.db
Wikipedia:   C:\Users\MDK\worldcup-rag-chatbot\data\raw\wikipedia
Synthetic:   C:\Users\MDK\worldcup-rag-chatbot\data\synthetic


## 1. Preview the raw CSVs

In [2]:
schedule_df = pd.read_csv(settings.paths.csvs / SCHEDULE_CSV)
print(f"schedule: {schedule_df.shape[0]} rows, {schedule_df.shape[1]} cols")
schedule_df.head()

schedule: 104 rows, 5 cols


,date,match_number,group,stadium,date_dt
0,"Thursday, 11 June 2026",Match 1,Group A,Mexico City Stadium,2026-06-11
1,"Thursday, 11 June 2026",Match 2,Group A,Estadio Guadalajara,2026-06-11
2,"Friday, 12 June 2026",Match 3,Group B,Toronto Stadium,2026-06-12
3,"Friday, 12 June 2026",Match 4,Group D,Los Angeles Stadium,2026-06-12
4,"Saturday, 13 June 2026",Match 5,Group C,Boston Stadium,2026-06-13


In [3]:
probs_df = pd.read_csv(settings.paths.csvs / PROBABILITIES_CSV)
print(f"probabilities: {probs_df.shape[0]} rows, {probs_df.shape[1]} cols")
probs_df.head()

probabilities: 72 rows, 13 cols


,group,home_team,away_team,date,tournament,home_elo,away_elo,elo_diff,home_injury_flag,away_injury_flag,p_home_win,p_draw,p_away_win
0,A,Mexico,South Africa,NaN,FIFA World Cup 2026 - Group,1890,1600.0,290.0,0,0,0.708931,0.157531,0.133538
1,A,Mexico,South Korea,NaN,FIFA World Cup 2026 - Group,1890,1755.0,135.0,0,0,0.547044,0.201465,0.251491
2,A,Mexico,UEFA_Playoff_D,NaN,FIFA World Cup 2026 - Group,1890,NaN,NaN,0,0,0.360000,0.280000,0.360000
3,A,South Africa,South Korea,NaN,FIFA World Cup 2026 - Group,1600,1755.0,-155.0,0,0,0.234344,0.193713,0.571944
4,A,South Africa,UEFA_Playoff_D,NaN,FIFA World Cup 2026 - Group,1600,NaN,NaN,0,0,0.360000,0.280000,0.360000


In [4]:
host_df = pd.read_csv(settings.paths.csvs / HOST_CITIES_CSV)
print(f"host_cities: {host_df.shape[0]} rows, {host_df.shape[1]} cols")
host_df.head()

host_cities: 16 rows, 6 cols


,id,city_name,country,venue_name,region_cluster,airport_code
0,1,Atlanta,USA,Mercedes-Benz Stadium,East,ATL
1,2,Boston,USA,Gillette Stadium,East,BOS
2,3,Dallas,USA,AT&T Stadium,Central,DAL
3,4,Houston,USA,NRG Stadium,Central,IAH
4,5,Kansas City,USA,Arrowhead Stadium,Central,MCI


## 2. Load all three CSVs into SQLite

In [5]:
row_counts = load_all_csvs()
row_counts

{'schedule': 104, 'probabilities': 72, 'host_cities': 16}

## 3. Generate bilingual text chunks

In [6]:
joined_rows = build_matches_full()
print(f"matches_full rows: {joined_rows}")

matches_full rows: 72


In [7]:
chunks = convert_all()
by_source: dict[str, int] = {}
for c in chunks:
    by_source[c["source"]] = by_source.get(c["source"], 0) + 1
print(f"Total chunks: {len(chunks)}")
print("By source:   ", by_source)

Total chunks: 264
By source:    {'schedule': 104, 'probabilities': 72, 'host_cities': 16, 'matches_full': 72}


### 5 sample English chunks

In [8]:
import random

random.seed(0)
for c in random.sample(chunks, 5):
    print(f"[{c['source']}] {c['text_en']}")

[matches_full] South Korea vs UEFA Playoff D on June 24, 2026 at Estadio BBVA, Monterrey.
[matches_full] Australia vs UEFA Playoff C on June 25, 2026 at Levi's Stadium, San Francisco Bay Area.
[schedule] Match 21 of Group L on June 17, 2026 at Toronto Stadium.
[probabilities] In Group E, Côte d’Ivoire vs Curaçao (FIFA World Cup 2026 - Group). Predicted probabilities: Côte d’Ivoire win 63.5%, draw 17.6%, Curaçao win 18.9%.
[matches_full] Croatia vs Ghana on June 23, 2026 at BMO Field, Toronto.


### 5 sample Arabic chunks

In [9]:
random.seed(1)
for c in random.sample(chunks, 5):
    print(f"[{c['source']}] {c['text_ar']}")

[schedule] المباراة 69 من المجموعة J في 27 يونيو 2026 على ملعب Kansas City Stadium.
[schedule] المباراة 33 من المجموعة E في 20 يونيو 2026 على ملعب Toronto Stadium.
[probabilities] في المجموعة E، ألمانيا ضد كوراساو (كأس العالم فيفا 2026 - دور المجموعات). الاحتمالات المتوقعة: فوز ألمانيا 81.7%، تعادل 13.4%، فوز كوراساو 4.9%.
[schedule] المباراة 61 من المجموعة I في 26 يونيو 2026 على ملعب Boston Stadium.
[matches_full] البرتغال ضد أوزبكستان في 17 يونيو 2026 في ملعب الأزتيكا، مكسيكو سيتي.


## 4. `matches_full` table preview

Group-stage rows from `schedule` zipped with the corresponding `probabilities` rows. 72 rows total (12 groups × 6 matches).

In [10]:
engine = create_engine(f"sqlite:///{settings.paths.sqlite_db}")
matches_df = pd.read_sql("SELECT * FROM matches_full", engine)
print(f"matches_full: {matches_df.shape[0]} rows, {matches_df.shape[1]} cols")
matches_df.head()

matches_full: 72 rows, 6 cols


,match_number,group,home_team,away_team,date,stadium
0,1,A,Mexico,South Africa,2026-06-11,Mexico City Stadium
1,2,A,Mexico,South Korea,2026-06-11,Estadio Guadalajara
2,25,A,Mexico,UEFA_Playoff_D,2026-06-18,Atlanta Stadium
3,28,A,South Africa,South Korea,2026-06-18,Estadio Guadalajara
4,53,A,South Africa,UEFA_Playoff_D,2026-06-24,Mexico City Stadium


## 5. Wikipedia corpus — word counts per file

In [11]:
wiki_files = sorted(settings.paths.wikipedia.glob("*.txt"))
rows = []
for path in wiki_files:
    text = path.read_text(encoding="utf-8")
    rows.append({
        "file": path.name,
        "chars": len(text),
        "words": len(text.split()),
    })
wiki_df = pd.DataFrame(rows)
print(f"{len(wiki_df)} files · {wiki_df['words'].sum():,} total words")
wiki_df

10 files · 55,522 total words


,file,chars,words
0,ar_2026_world_cup.txt,10055,1689
1,ar_fifa_world_cup.txt,80778,13757
2,ar_morocco_team.txt,10565,1779
3,ar_saudi_arabia_team.txt,14044,2347
4,ar_world_cup_history.txt,54070,9031
5,en_2026_world_cup.txt,54984,8826
6,en_fifa_world_cup.txt,37132,6121
7,en_morocco_team.txt,21045,3462
8,en_saudi_arabia_team.txt,14894,2463
9,en_world_cup_history.txt,35785,6047


## 6. Synthetic Q&A samples

Produced by `python -m src.training.qa_generator` (use `--dry-run` for the placeholder pairs shown here; the full run uses the Anthropic API).

In [12]:
def _load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        print(f"(missing: {path.name} — run `python -m src.training.qa_generator --dry-run`)")
        return []
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

qa_ar = _load_jsonl(settings.paths.synthetic / "qa_arabic.jsonl")
qa_en = _load_jsonl(settings.paths.synthetic / "qa_english.jsonl")
print(f"Loaded {len(qa_ar)} Arabic pairs and {len(qa_en)} English pairs.")

Loaded 10 Arabic pairs and 10 English pairs.


### 5 English Q&A pairs

In [13]:
for pair in qa_en[:5]:
    print(f"[{pair['source']}]")
    print(f"  Q: {pair['question']}")
    print(f"  A: {pair['answer']}")

[schedule]
  Q: What does the schedule record say?
  A: Match 42 of Group I on June 22, 2026 at Philadelphia Stadium.
[matches_full]
  Q: What does the matches_full record say?
  A: South Africa vs UEFA Playoff D on June 24, 2026 at Estadio Azteca, Mexico City.
[host_cities]
  Q: What does the host_cities record say?
  A: Mercedes-Benz Stadium is in Atlanta, USA (airport code ATL, East region cluster).
[probabilities]
  Q: What does the probabilities record say?
  A: In Group H, Spain vs Cape Verde (FIFA World Cup 2026 - Group). Predicted probabilities: Spain win 81.4%, draw 13.5%, Cape Verde win 5.1%.
[matches_full]
  Q: What does the matches_full record say?
  A: Mexico vs UEFA Playoff D on June 18, 2026 at Mercedes-Benz Stadium, Atlanta.


### 5 Arabic Q&A pairs

In [14]:
for pair in qa_ar[:5]:
    print(f"[{pair['source']}]")
    print(f"  Q: {pair['question']}")
    print(f"  A: {pair['answer']}")

[schedule]
  Q: ماذا يقول سجل schedule؟
  A: المباراة 42 من المجموعة I في 22 يونيو 2026 على ملعب Philadelphia Stadium.
[matches_full]
  Q: ماذا يقول سجل matches_full؟
  A: جنوب أفريقيا ضد ملحق يويفا D في 24 يونيو 2026 في ملعب الأزتيكا، مكسيكو سيتي.
[host_cities]
  Q: ماذا يقول سجل host_cities؟
  A: ملعب مرسيدس بنز يقع في أتلانتا، الولايات المتحدة (رمز المطار ATL، منطقة الشرق).
[probabilities]
  Q: ماذا يقول سجل probabilities؟
  A: في المجموعة H، إسبانيا ضد الرأس الأخضر (كأس العالم فيفا 2026 - دور المجموعات). الاحتمالات المتوقعة: فوز إسبانيا 81.4%، تعادل 13.5%، فوز الرأس الأخضر 5.1%.
[matches_full]
  Q: ماذا يقول سجل matches_full؟
  A: المكسيك ضد ملحق يويفا D في 18 يونيو 2026 في ملعب مرسيدس بنز، أتلانتا.
